# Turbofan RUL Tahmini — Bölüm 3
## İlk Model Karşılaştırması ve Aday Model Seçimi (Aşama 8–9)

Bu notebook, Bölüm 2'de kurulan **sızıntısız Pipeline** ve **motor bazlı GroupKFold** deney tasarımını kullanarak 8 farklı algoritmayı **aynı koşullar altında** karşılaştırır ve hiperparametre optimizasyonuna (Aşama 10) girecek aday modeli gerekçeli biçimde seçer.

> ⚠️ Yalnızca `train_FD001.txt` kullanılır. Test dosyaları Aşama 12'ye kadar açılmaz.

> ⏱️ Colab'da (2 çekirdek) toplam çalışma süresi ≈ 6–8 dakika. En uzun süren modeller Random Forest (~2.5 dk) ve SVR (~2 dk).

## 0. Kurulum
### 0.1 Google Drive bağlantısı
Tüm notebook'lar aynı Drive klasöründe çalışır. Böylece `train_FD001.txt` bir kez yüklenir ve notebook'ların ürettiği dosyalar (`project_config.json`, sonuç tabloları) kaybolmaz.

**Hazırlık:** Drive'da `BTK_RUL` adında bir klasör açıp `train_FD001.txt` dosyasını içine koyun. (Bölüm 2'yi de bu klasörde çalıştırdıysanız `project_config.json` ve `results_stage5_7.csv` de burada olacaktır.)

In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/BTK_RUL"   # klasör adını değiştirdiyseniz burayı güncelleyin

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BASE_DIR, exist_ok=True)
    os.chdir(BASE_DIR)
except ImportError:
    print("Colab dışında çalışılıyor → mevcut klasör kullanılacak.")

print("Çalışma klasörü:", os.getcwd())
print("Klasördeki dosyalar:", sorted(os.listdir(".")))
if not os.path.exists("train_FD001.txt"):
    raise FileNotFoundError("train_FD001.txt bulunamadı. Dosyayı Drive'daki BTK_RUL klasörüne yükleyip bu hücreyi tekrar çalıştırın.")

### 0.2 Ortak yardımcı kod
Bölüm 2'deki `rul_utils.py`'nin genişletilmiş sürümü. Aşama 8–9 için üç yeni ölçüt eklendi: arızaya yakın bölge hatası (`near50_RMSE`), geç tahmin oranı (`late10_pct`) ve aşırı öğrenme kontrolü için eğitim hatası (`train_clip_RMSE`). Bölüm 2'deki fonksiyonların davranışı değişmedi, yani sonuçlar Bölüm 2 ile birebir karşılaştırılabilir.

In [ ]:
%%writefile rul_utils.py
"""rul_utils.py — NASA C-MAPSS RUL projesi için ortak yardımcılar.
Notebook'lar ve Streamlit uygulaması aynı kodu kullanır (tekrarlanabilirlik)."""
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

INDEX_COLS = ["unit", "cycle"]
SETTING_COLS = ["op1", "op2", "op3"]
SENSOR_COLS = [f"s{i}" for i in range(1, 22)]
COLS = INDEX_COLS + SETTING_COLS + SENSOR_COLS
FEATURES_14 = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12",
               "s13", "s14", "s15", "s17", "s20", "s21"]


def load_cmapss(path):
    """Boşlukla ayrılmış, başlıksız C-MAPSS dosyasını okur."""
    df = pd.read_csv(path, sep=r"\s+", header=None).iloc[:, :26]
    df.columns = COLS
    return df.sort_values(["unit", "cycle"]).reset_index(drop=True)


def add_rul(df):
    """Eğitim verisi için gerçek (kırpılmamış) RUL: motorun son döngüsü - mevcut döngü."""
    df = df.copy()
    df["RUL"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
    return df


def _rolling_slope(x, window):
    """Geçmiş `window` döngüdeki doğrusal eğim (yalnızca geçmişe bakar)."""
    s = pd.Series(np.asarray(x, dtype=float))
    t = pd.Series(np.arange(len(s), dtype=float))
    cov = s.rolling(window, min_periods=2).cov(t)
    var = t.rolling(window, min_periods=2).var()
    return (cov / var).fillna(0.0).to_numpy()


class RollingFeatures(BaseEstimator, TransformerMixin):
    """Motor bazında, yalnızca GEÇMİŞ döngüleri kullanan kayan pencere özellikleri.

    - Girdi: 'unit', 'cycle' ve sensör sütunlarını içeren DataFrame (motor içinde döngü sırasına göre).
    - Çıktı: ham sensörler (+ isteğe bağlı) kayan ortalama, standart sapma ve eğim.
    - 'unit' ve 'cycle' çıktıya ALINMAZ (özellik değil, yalnızca gruplama/sıra bilgisi).
    - fit() hiçbir şey öğrenmez (stateless): bir satırın özellikleri yalnızca kendi motorunun
      o ana kadarki kayıtlarından hesaplanır → başka motorlardan veya gelecekten bilgi sızmaz.
    """

    def __init__(self, sensors=tuple(FEATURES_14), window=10, stats=("mean", "std", "slope"), keep_raw=True):
        self.sensors = sensors
        self.window = window
        self.stats = stats
        self.keep_raw = keep_raw

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        if not X.groupby("unit")["cycle"].apply(lambda c: c.is_monotonic_increasing).all():
            raise ValueError("Her motorun satırları döngü sırasına göre sıralı olmalı.")
        sensors, w = list(self.sensors), int(self.window)
        g = X.groupby("unit", sort=False)
        parts = []
        if self.keep_raw:
            parts.append(X[sensors].reset_index(drop=True))
        if "mean" in self.stats:
            m = g[sensors].transform(lambda c: c.rolling(w, min_periods=1).mean())
            parts.append(m.add_suffix(f"_mean{w}").reset_index(drop=True))
        if "std" in self.stats:
            sd = g[sensors].transform(lambda c: c.rolling(w, min_periods=2).std()).fillna(0.0)
            parts.append(sd.add_suffix(f"_std{w}").reset_index(drop=True))
        if "slope" in self.stats:
            sl = g[sensors].transform(lambda c: _rolling_slope(c, w))
            parts.append(sl.add_suffix(f"_slope{w}").reset_index(drop=True))
        out = pd.concat(parts, axis=1)
        out.index = X.index
        return out

    def get_feature_names_out(self, input_features=None):
        names = list(self.sensors) if self.keep_raw else []
        for st in ("mean", "std", "slope"):
            if st in self.stats:
                names += [f"{s}_{st}{self.window}" for s in self.sensors]
        return np.array(names)


def nasa_score(y_true, y_pred):
    """PHM08 skor fonksiyonu. d = tahmin - gerçek.
    Geç tahmin (d > 0, ömrü fazla tahmin etmek) daha ağır cezalandırılır."""
    d = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1, np.exp(d / 10.0) - 1)))


def regression_metrics(y_true, y_pred, prefix=""):
    return {
        f"{prefix}RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{prefix}MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}R2": float(r2_score(y_true, y_pred)),
        f"{prefix}NASA": nasa_score(y_true, y_pred),
    }


def test_like_rows(df, idx, n_cuts=10, min_cycle=31, max_rul=150, seed=42):
    """Validation motorlarından resmi test protokolünü taklit eden satırlar seçer:
    her motor için rastgele `n_cuts` kesim noktası; kesim anındaki GERÇEK RUL ile değerlendirilir.
    - min_cycle: kayan pencerelerin dolması için gereken minimum geçmiş.
    - max_rul: bakım karar ufku. Kesim noktaları gerçek RUL <= max_rul olan döngülerden seçilir
      (EDA: bozulma ömrün ikinci yarısında görünür hale geliyor; daha erken döngülerde RUL
      sensörlerden ayırt edilemez ve bakım kararı açısından önemsizdir)."""
    rng = np.random.default_rng(seed)
    sub = df.loc[idx]
    chosen = []
    for _, d in sub.groupby("unit"):
        cand = d.index[(d["cycle"] >= min(min_cycle, d["cycle"].max())) & (d["RUL"] <= max_rul)]
        chosen.extend(rng.choice(cand, size=min(n_cuts, len(cand)), replace=False))
    return np.sort(np.array(chosen))


def cv_evaluate(model, df, cap, cv, feature_cols=None, n_cuts=10, max_rul=150, seed=42, name=None,
                train_metrics=False, model_size=False, return_preds=False):
    """Motor bazlı CV ile değerlendirme.
    - Eğitim hedefi: RUL üst sınırla kırpılır (cap=None ise kırpılmaz).
    - 'clip_' metrikleri: validation motorlarının tüm döngüleri, kırpılmış hedefe göre.
    - 'test_' metrikleri: validation motorlarında rastgele kesim noktaları, GERÇEK RUL'a göre
      ve bakım karar ufku (gerçek RUL <= max_rul) içinde → model ve cap seçiminde birincil ölçüt.
    - train_metrics=True: eğitim fold'undaki RMSE (aşırı öğrenme kontrolü).
    - model_size=True: eğitilmiş modelin bellekteki (pickle) boyutu, MB.
    - return_preds=True: (sonuç_tablosu, kesim_noktası_tahminleri) döndürür (hata analizi için).
    """
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_true = df["RUL"].to_numpy()
    y_fit = y_true if cap is None else np.minimum(y_true, cap)
    rows, preds = [], []
    for k, (tr, va) in enumerate(cv.split(X, y_fit, groups=df["unit"])):
        m = clone(model)
        t0 = time.perf_counter()
        m.fit(X.iloc[tr], y_fit[tr])
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        p = np.clip(m.predict(X.iloc[va]), 0, None)
        pred_s = time.perf_counter() - t0
        res = {"fold": k + 1, "fit_s": fit_s, "pred_s": pred_s}
        res.update(regression_metrics(y_fit[va], p, "clip_"))
        cut = test_like_rows(df, df.index[va], n_cuts=n_cuts, max_rul=max_rul, seed=seed + k)
        pos = pd.Series(np.arange(len(va)), index=df.index[va]).loc[cut].to_numpy()
        res.update(regression_metrics(y_true[cut], p[pos], "test_"))
        # Arızaya yakın bölge (gerçek RUL <= 50): bakım kararının en kritik olduğu dönem
        near = y_true[cut] <= 50
        res["near50_RMSE"] = float(np.sqrt(np.mean((p[pos][near] - y_true[cut][near]) ** 2))) if near.any() else np.nan
        # Geç tahmin oranı: ömrü 10 döngüden fazla FAZLA tahmin etme (güvenlik açısından tehlikeli)
        res["late10_pct"] = float(np.mean(p[pos] - y_true[cut] > 10) * 100)
        if train_metrics:
            p_tr = np.clip(m.predict(X.iloc[tr]), 0, None)
            res["train_clip_RMSE"] = float(np.sqrt(mean_squared_error(y_fit[tr], p_tr)))
        if model_size:
            import pickle
            res["size_MB"] = len(pickle.dumps(m)) / 1e6
        rows.append(res)
        if return_preds:
            preds.append(pd.DataFrame({"model": name, "fold": k + 1, "unit": df.loc[cut, "unit"].to_numpy(),
                                       "cycle": df.loc[cut, "cycle"].to_numpy(),
                                       "RUL_true": y_true[cut], "RUL_pred": p[pos]}))
    out = pd.DataFrame(rows)
    if name is not None:
        out.insert(0, "model", name)
    if return_preds:
        return out, pd.concat(preds, ignore_index=True)
    return out


def summarize(cv_df, by="model"):
    cols = [c for c in ["test_RMSE", "test_MAE", "test_R2", "test_NASA", "near50_RMSE", "late10_pct",
                        "clip_RMSE", "train_clip_RMSE", "fit_s", "pred_s", "size_MB"] if c in cv_df.columns]
    agg = cv_df.groupby(by)[cols].agg(["mean", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    return agg.sort_values("test_RMSE_mean")


def oof_predictions(model, df, cap, cv, feature_cols=None):
    """Out-of-fold tahminler: her satır, motoru validation'dayken eğitilen modelden tahmin edilir."""
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_fit = df["RUL"].to_numpy() if cap is None else np.minimum(df["RUL"].to_numpy(), cap)
    oof = np.zeros(len(df))
    for tr, va in cv.split(X, y_fit, groups=df["unit"]):
        m = clone(model).fit(X.iloc[tr], y_fit[tr])
        oof[va] = np.clip(m.predict(X.iloc[va]), 0, None)
    return oof

In [ ]:
import warnings, json, time, platform
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from rul_utils import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

### 0.3 Bölüm 2 kararlarının yüklenmesi

In [ ]:
DEFAULT_CONFIG = {"seed": 42, "cap": 125, "window": 30, "stats": ["mean", "std", "slope"],
                  "eval_max_rul": 150, "eval_n_cuts": 10}
if os.path.exists("project_config.json"):
    config = {**DEFAULT_CONFIG, **json.load(open("project_config.json"))}
    print("project_config.json okundu (Bölüm 2 kararları).")
else:
    config = DEFAULT_CONFIG.copy()
    print("UYARI: project_config.json bulunamadı → Bölüm 2'de seçilen varsayılan değerler kullanılıyor.")

SEED, CAP, WINDOW = config["seed"], config["cap"], config["window"]
STATS = tuple(config["stats"])
np.random.seed(SEED)
print(f"seed={SEED} | cap={CAP} | pencere={WINDOW} | özellikler={STATS} | karar ufku={config['eval_max_rul']}")

df = add_rul(load_cmapss("train_FD001.txt"))
cv = GroupKFold(n_splits=5)
print("Eğitim verisi:", df.shape, "| motor:", df["unit"].nunique())

## Aşama 8 — İlk Model Karşılaştırması
### 8.1 Adil karşılaştırma koşulları

Yönergenin "aynı veri split ve uygun preprocessing" şartı için **tüm modellerde her şey ortak, değişen tek şey algoritma**:

| Ortak koşul | Değer |
|---|---|
| Veri | `train_FD001.txt`, 100 motor |
| Bölme | Aynı `GroupKFold(5)` nesnesi → her model aynı 5 fold'da, aynı motorlarla |
| Değerlendirme noktaları | Aynı test-benzeri kesim noktaları (her fold için sabit seed) |
| Özellikler | Aynı `RollingFeatures(window=30)` → 56 özellik |
| Hedef | Aynı parçalı doğrusal RUL, cap = 125 |
| Özellik ölçekleme | `StandardScaler`, her fold'un eğitim verisinde fit |
| **Hedef ölçekleme** | `TransformedTargetRegressor(StandardScaler)` — aşağıda açıklandı |
| Hiperparametre | **Tuning yok.** Makul varsayılanlar; tuning yalnızca seçilen aday(lar) için Aşama 10'da |

**Neden hedef (RUL) de ölçekleniyor?** SVR'nin hata toleransı (`epsilon = 0.1`) ve MLP'nin öğrenme oranı, hedefin standart ölçekte (ortalama 0, std 1) olduğu varsayımıyla belirlenmiştir. RUL 0–125 aralığında kalırsa bu iki model varsayılan ayarlarla yapay olarak kötü sonuç verir ve karşılaştırma adil olmaz. Hedef ölçekleme doğrusal modellerin, ağaçların ve KNN'nin tahminlerini değiştirmez, dolayısıyla diğer modellere bir avantaj ya da dezavantaj getirmez. Ölçekleyici her fold'da **yalnızca eğitim hedefinden** öğrenilir, tahminler otomatik olarak orijinal ölçeğe döndürülür.

In [ ]:
def make_model(estimator, window=WINDOW, stats=STATS):
    """Ortak pipeline: ham sensör kaydı → kayan özellikler → ölçekleme → model (hedef de ölçeklenir)."""
    pipe = Pipeline([("features", RollingFeatures(window=window, stats=stats)),
                     ("scaler", StandardScaler()),
                     ("model", estimator)])
    return TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler())

### 8.2 Model seçimi ve gerekçesi

Yönerge en az 3 model istiyor. Problemin yapısını farklı açılardan yakalayabilecek **5 farklı model ailesinden 8 algoritma** seçildi:

| Model | Aile | Neden bu probleme uygun? | Kullanılan ayarlar (tuning yok) |
|---|---|---|---|
| **Ridge** | Doğrusal + L2 | Baseline'ın düzenlileştirilmiş hali; çok korelasyonlu sensörlerin (`s9`–`s14`, r = 0.96) katsayılarını kararlı hale getirir | `alpha=1.0` (varsayılan) |
| **Lasso** | Doğrusal + L1 | Gereksiz özellikleri sıfırlayarak özellik seçimi yapar → 56 özelliğin kaçının gerçekten gerekli olduğunu gösterir | `alpha=0.01` ¹ |
| **KNN** | Örnek tabanlı | "Benzer sensör profiline sahip motorların kalan ömrü de benzerdir" sezgisi; parametrik varsayım yok | `n_neighbors=10`, `weights="distance"` ² |
| **SVR (RBF)** | Kernel | Doğrusal olmayan ilişkileri düzgün (smooth) bir fonksiyonla modeller; uç değerlere görece dayanıklı | Varsayılan (`C=1`, `epsilon=0.1`, `gamma="scale"`) |
| **Decision Tree** | Tek ağaç | En yorumlanabilir doğrusal olmayan model; topluluk (ensemble) yöntemlerinin kazancını göstermek için referans | Varsayılan (derinlik sınırı yok) |
| **Random Forest** | Bagging ensemble | Bozulmanın eşik benzeri, doğrusal olmayan yapısını yakalar; ağaçların ortalaması gürültüye dayanıklıdır | `n_estimators=100` ³ |
| **HistGradientBoosting** | Boosting ensemble | Tablo verisinde genelde en güçlü aile; hatalar üzerine sıralı öğrenme yapar; büyük veride hızlıdır | Varsayılan |
| **MLP** | Yapay sinir ağı | Esnek doğrusal olmayan fonksiyon; literatürdeki derin öğrenme yaklaşımlarının basit bir temsilcisi | `(64, 32)` gizli katman, `early_stopping=True` ⁴ |

¹ Lasso'nun varsayılan `alpha=1.0` değeri, standartlaştırılmış hedefte tüm katsayıları sıfırlar (model sabit tahmine düşer). 0.01, L1'in etkisini gösterecek ama modeli öldürmeyecek bir değer.
² `weights="distance"`: yakın komşular daha çok etki eder; k=5 (varsayılan) gürültülü sensörlerde çok oynak olduğu için 10 kullanıldı.
³ 200 ağaçla yapılan ön denemede sonuç neredeyse aynıydı (RMSE farkı < 0.1) ama süre iki katıydı; Colab süresini makul tutmak için 100 seçildi.
⁴ `early_stopping`: verinin %10'unu ayırıp doğrulama hatası iyileşmeyince eğitimi durdurur → aşırı öğrenmeye karşı standart önlem. `max_iter=500`.

**Neden XGBoost / LightGBM burada yok?** Bunlar HistGradientBoosting ile aynı aileden (gradient boosting) ve Aşama 11'deki **"güçlü rakipler"** için ayrıldı. Aşama 8'in amacı farklı model ailelerini karşılaştırıp hangi ailenin bu probleme uygun olduğunu görmek.

In [ ]:
models = {
    "Ridge": make_model(Ridge(alpha=1.0)),
    "Lasso": make_model(Lasso(alpha=0.01, max_iter=5000)),
    "KNN": make_model(KNeighborsRegressor(n_neighbors=10, weights="distance", n_jobs=-1)),
    "SVR (RBF)": make_model(SVR(kernel="rbf")),
    "DecisionTree": make_model(DecisionTreeRegressor(random_state=SEED)),
    "RandomForest": make_model(RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)),
    "HistGradientBoosting": make_model(HistGradientBoostingRegressor(random_state=SEED)),
    "MLP": make_model(MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=500, random_state=SEED)),
}
# Referans: Aşama 7 baseline'ları (aynı koşullarda yeniden hesaplanır → tabloda yan yana)
baselines = {
    "[baseline] Dummy (ortalama)": make_model(DummyRegressor(strategy="mean"), stats=()),
    "[baseline] LinearRegression (ham sensör)": make_model(LinearRegression(), stats=()),
}

### 8.3 Deney: 5-fold GroupKFold ile değerlendirme

In [ ]:
all_res, all_preds = [], []
for name, model in {**baselines, **models}.items():
    t0 = time.perf_counter()
    r, p = cv_evaluate(model, df, CAP, cv, name=name, n_cuts=config["eval_n_cuts"], max_rul=config["eval_max_rul"],
                       seed=SEED, train_metrics=True, model_size=True, return_preds=True)
    all_res.append(r); all_preds.append(p)
    print(f"{name:42s} test-benzeri RMSE = {r['test_RMSE'].mean():6.2f} ± {r['test_RMSE'].std():.2f}   ({time.perf_counter()-t0:5.1f} sn)")

res = pd.concat(all_res, ignore_index=True)
preds = pd.concat(all_preds, ignore_index=True)
res.to_csv("results_stage8_folds.csv", index=False)
preds.to_csv("results_stage8_cut_predictions.csv", index=False)

### 8.4 Sonuç tablosu

| Sütun | Anlamı | Yön |
|---|---|---|
| `test_RMSE` | Karar ufkunda (gerçek RUL ≤ 150) test-benzeri kesim noktalarında RMSE — **birincil metrik** | ↓ |
| `test_MAE`, `test_R2` | Aynı noktalarda MAE ve R² | ↓ / ↑ |
| `test_NASA` | PHM08 skoru — ömrü fazla tahmin etmeyi (geç tahmin) üstel olarak cezalandırır | ↓ |
| `near50_RMSE` | Yalnızca gerçek RUL ≤ 50 olan noktalarda RMSE — bakım kararının en kritik olduğu dönem | ↓ |
| `late10_pct` | Ömrü 10 döngüden fazla **fazla** tahmin edilen noktaların yüzdesi — güvenlik riski | ↓ |
| `clip_RMSE` / `train_clip_RMSE` | Kırpılmış hedefe göre validation / eğitim RMSE'si → aradaki fark aşırı öğrenme göstergesi | ↓ |
| `fit_s`, `pred_s`, `size_MB` | Fold başına eğitim süresi, tahmin süresi, eğitilmiş modelin boyutu | ↓ |

In [ ]:
summary = summarize(res)
main_cols = ["test_RMSE_mean", "test_RMSE_std", "test_MAE_mean", "test_R2_mean", "test_NASA_mean",
             "near50_RMSE_mean", "late10_pct_mean", "clip_RMSE_mean", "train_clip_RMSE_mean",
             "fit_s_mean", "pred_s_mean", "size_MB_mean"]
summary = summary[main_cols]
summary.to_csv("results_stage8_summary.csv")
display(summary.round(2).style.background_gradient(subset=["test_RMSE_mean", "test_NASA_mean", "near50_RMSE_mean"], cmap="RdYlGn_r")
        .format(precision=2))

### 8.5 Görselleştirme

In [ ]:
order = summary.index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(18, 6))
sns.boxplot(data=res, y="model", x="test_RMSE", order=order, ax=ax[0], color="lightsteelblue")
sns.stripplot(data=res, y="model", x="test_RMSE", order=order, ax=ax[0], color="k", size=4)
ax[0].axvline(summary.loc["[baseline] LinearRegression (ham sensör)", "test_RMSE_mean"], c="r", ls="--", label="ana baseline")
ax[0].set_title("Test-benzeri RMSE — 5 fold dağılımı (düşük daha iyi)"); ax[0].set_ylabel(""); ax[0].legend()

s2 = summary.drop(index="[baseline] Dummy (ortalama)")
ax[1].barh(s2.index, s2["test_NASA_mean"], xerr=res.groupby("model")["test_NASA"].std().reindex(s2.index), color="salmon")
ax[1].invert_yaxis(); ax[1].set_xscale("log"); ax[1].set_title("NASA skoru (log ölçek, düşük daha iyi)")
plt.tight_layout(); plt.show()

**Yorum:**
- Modeller belirgin biçimde **iki gruba** ayrılıyor. **Güçlü doğrusal olmayan modeller** (HistGradientBoosting, Random Forest, SVR, MLP) 16–18 RMSE bandında. **Doğrusal modeller, KNN ve tek ağaç** ise 20–22.5 bandında, yani baseline'a yakın.
- **HistGradientBoosting** hem en düşük ortalama RMSE'ye (~16.0) hem de en düşük fold varyansına (std ~1.1) sahip. En kötü fold'u bile diğer modellerin çoğunun ortalamasından iyi.
- Ridge ve Lasso, ham LR baseline'ından yalnızca kayan özellikler sayesinde iyi (~21.3–21.5). L2/L1 düzenlileştirme ek kazanç sağlamıyor; bu da doğrusal modellerin sınırına ulaştığını gösteriyor.
- **Decision Tree** RMSE'de baseline'dan bile kötü (~22.5) ve NASA skoru en yüksek modellerden. Tek ağaç çok oynak; Random Forest ve boosting'in kazancı tam olarak bu oynaklığı ortalamalarından geliyor.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
s3 = summary.drop(index="[baseline] Dummy (ortalama)")
ax.scatter(s3["fit_s_mean"], s3["test_RMSE_mean"], s=np.clip(s3["size_MB_mean"], 0.05, None) * 8 + 40, alpha=.7)
for n, r in s3.iterrows():
    ax.annotate(n.replace("[baseline] ", "BL: "), (r["fit_s_mean"], r["test_RMSE_mean"]), xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Fold başına eğitim süresi (sn, log)"); ax.set_ylabel("Test-benzeri RMSE")
ax.set_title("Performans – maliyet dengesi (nokta büyüklüğü ∝ model boyutu)"); plt.show()

**Yorum:** Sol alt köşe (hızlı ve isabetli) ideal bölge. **HistGradientBoosting tam bu köşede**: ~1–2 saniyede eğitiliyor ve model boyutu ~0.5 MB. **Random Forest** isabette ona yakın ama eğitimi ~20 kat uzun sürüyor ve model boyutu çok büyük (~80 MB). Bu, Streamlit Cloud'da deploy açısından da bir dezavantaj (bellek ve yükleme süresi). **SVR**'nin eğitimi orta seviyede ama tahmin süresi en uzun olanlardan; kernel hesabı tüm destek vektörleriyle yapıldığı için veri büyüdükçe hızla yavaşlar.

### 8.6 Aşırı öğrenme (overfitting) kontrolü

In [ ]:
ov = summary[["train_clip_RMSE_mean", "clip_RMSE_mean"]].drop(index="[baseline] Dummy (ortalama)").copy()
ov["fark"] = ov["clip_RMSE_mean"] - ov["train_clip_RMSE_mean"]
ax = ov[["train_clip_RMSE_mean", "clip_RMSE_mean"]].rename(columns={"train_clip_RMSE_mean": "eğitim", "clip_RMSE_mean": "validation"}).plot(kind="barh", figsize=(10, 6))
ax.invert_yaxis(); ax.set_xlabel("RMSE (kırpılmış hedef)"); ax.set_title("Eğitim vs validation hatası"); plt.show()
display(ov.round(2))

**Yorum:**
- **KNN ve Decision Tree** eğitim verisini ezberliyor: eğitim hatası ~0, validation hatası ~18–21. KNN `weights="distance"` ile her eğitim noktasını kendisiyle tahmin ediyor; tek ağaç ise derinlik sınırı olmadan her yaprakta tek örnek bırakıyor.
- **Random Forest** da eğitimde ~2.6 RMSE'ye iniyor (tek tek ağaçlar ezberliyor), ama ortalama alındığı için validation'da iyi genelleşiyor. Bagging'in beklenen davranışı bu.
- **HistGradientBoosting**'de fark orta düzeyde (~5.7 → ~13.2). Bu bir miktar aşırı öğrenme olduğunu ve düzenlileştirme parametrelerinin (`max_depth`, `min_samples_leaf`, `l2_regularization`, öğrenme oranı) tuning ile iyileştirilebileceğini gösteriyor. Aşama 10 için önemli bir ipucu.
- **Doğrusal modellerde** eğitim ve validation hataları neredeyse eşit (~19). Bu aşırı öğrenme değil, **yetersiz öğrenme (underfitting)**: model bozulmanın doğrusal olmayan yapısını temsil edemiyor.

### 8.7 Arızaya yakın bölgede hata analizi

Bakım kararı açısından en kritik dönem motorun arızaya yaklaştığı son döngüler. Burada **ömrü fazla tahmin etmek** (motoru arızaya kadar çalıştırmak) en tehlikeli hata. Aşağıdaki analiz, 8.3'te kaydedilen kesim noktası tahminlerini kullanıyor; modeller tekrar eğitilmiyor.

In [ ]:
top = [m for m in summary.index if not m.startswith("[baseline] Dummy")][:5] + ["[baseline] LinearRegression (ham sensör)"]
pp = preds[preds["model"].isin(top)].copy()
pp["hata"] = pp["RUL_pred"] - pp["RUL_true"]
pp["RUL_aralığı"] = pd.cut(pp["RUL_true"], [0, 25, 50, 75, 100, 125, 150], include_lowest=True)

fig, ax = plt.subplots(1, 2, figsize=(19, 5.5))
rmse_bin = pp.groupby(["RUL_aralığı", "model"], observed=True)["hata"].apply(lambda e: np.sqrt(np.mean(e**2))).unstack()[top]
rmse_bin.plot(marker="o", ax=ax[0]); ax[0].set_ylabel("RMSE"); ax[0].set_title("Gerçek RUL aralığına göre RMSE")
sns.boxplot(data=pp, x="RUL_aralığı", y="hata", hue="model", hue_order=top, showfliers=False, ax=ax[1])
ax[1].axhline(0, c="k", lw=1); ax[1].set_ylabel("Hata = tahmin − gerçek  (+ : geç tahmin, tehlikeli)")
ax[1].set_title("RUL aralığına göre hata dağılımı"); ax[1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()
display(rmse_bin.round(1))

**Yorum:**
- **Hata, RUL aralığına göre üç bölgeye ayrılıyor:**
  - **Arızaya yakın bölge (RUL ≤ 50):** Hata en düşük. Güçlü modellerde RUL ≤ 25 için ~4–8, 25–50 için ~12–15 RMSE. Bozulma belirginleştiğinde sensörler çok bilgilendirici hale geliyor.
  - **Geçiş bölgesi (RUL 50–100):** Hata en yüksek (~18–21). Bu, bozulmanın **başladığı** dönem; her motorda farklı zamanda başladığı için "bozulma başladı mı, ne kadar ilerledi?" sorusu en belirsiz burada.
  - **Sağlıklı bölge (RUL > 100):** RUL 100–125 aralığında hata yeniden düşüyor (~15). Modeller burada ~125'e yakın tahmin yapıyor ve gerçek değer de buna yakın. **RUL > 125** noktalarında ise hiçbir model doğru tahmin yapamıyor, çünkü eğitim hedefi 125'te sabitlendi. Bu bilinçli bir **tasarım bedeli**: sağlıklı motorlarda kalan ömrü eksik tahmin etmek temkinli bir hata ve bakım kararı açısından zararsız.
- Arızaya yakın bölgede genel olarak **HistGradientBoosting en düşük hataya sahip** (`near50_RMSE` ≈ 8.9; RUL 25–50 aralığında açık ara en iyi, 0–25 aralığında Random Forest ile başa baş). Baseline ise bu bölgede çok daha yüksek hata veriyor (~18–25) ve sistematik olarak **geç tahmin** yapıyor (kutular sıfırın üstünde; geç tahmin oranı ~%45, HGB'de ~%25).

### 8.8 Eşleştirilmiş (paired) fold karşılaştırması
Tüm modeller aynı fold'larda değerlendirildiği için fold bazında doğrudan karşılaştırılabilir. Ortalamalar arasındaki küçük farkların tesadüf mü yoksa tutarlı mı olduğunu görmek için en iyi modelin diğerlerini kaç fold'da geçtiğine bakıyoruz.

In [ ]:
best = summary.index[0]
piv = res.pivot(index="fold", columns="model", values="test_RMSE")
paired = pd.DataFrame({
    "ort_fark (rakip − en iyi)": piv.drop(columns=best).sub(piv[best], axis=0).mean(),
    "std_fark": piv.drop(columns=best).sub(piv[best], axis=0).std(),
    f"{best} daha iyi (fold / 5)": (piv.drop(columns=best).gt(piv[best], axis=0)).sum(),
}).sort_values("ort_fark (rakip − en iyi)")
print("En iyi model:", best)
display(paired.round(2))
display(piv.round(2))

**Yorum:** HistGradientBoosting en yakın iki rakibi olan Random Forest ve SVR'ye karşı ortalama farkı küçük (~0.4–0.6 RMSE) ama **tutarlı**: 5 fold'un çoğunda daha iyi. Doğrusal modellere, KNN'ye ve tek ağaca karşı ise **5 fold'un tamamında** ve büyük bir farkla önde. 5 fold ile resmi bir istatistiksel test (ör. Wilcoxon) güvenilir sonuç vermeyeceği için (en küçük olası p-değeri ≈ 0.06), karar fold tutarlılığı ve ek ölçütlerle (8.4–8.7) birlikte verilmektedir.

### 8.9 Performans, hız, karmaşıklık ve yorumlanabilirlik karşılaştırması

In [ ]:
qual = pd.DataFrame({
    "Yorumlanabilirlik": {"Ridge": "Yüksek (katsayılar)", "Lasso": "Yüksek (seyrek katsayılar)", "KNN": "Orta (benzer motorlar gösterilebilir)",
                          "SVR (RBF)": "Düşük", "DecisionTree": "Yüksek (kurallar)", "RandomForest": "Orta (özellik önemi)",
                          "HistGradientBoosting": "Orta (özellik önemi, SHAP)", "MLP": "Düşük"},
    "Karmaşıklık": {"Ridge": "Düşük", "Lasso": "Düşük", "KNN": "Düşük (eğitim yok, tahmin pahalı)", "SVR (RBF)": "Yüksek (O(n²) bellek/zaman)",
                    "DecisionTree": "Düşük", "RandomForest": "Yüksek (100 derin ağaç)", "HistGradientBoosting": "Orta (100 sığ ağaç)", "MLP": "Orta"},
    "Ölçeklenebilirlik": {"Ridge": "Çok iyi", "Lasso": "Çok iyi", "KNN": "Zayıf (tahmin süresi veriyle büyür)", "SVR (RBF)": "Zayıf",
                          "DecisionTree": "Çok iyi", "RandomForest": "Orta", "HistGradientBoosting": "Çok iyi (histogram tabanlı)", "MLP": "İyi"},
})
comp = summary.loc[list(models.keys()), ["test_RMSE_mean", "test_NASA_mean", "near50_RMSE_mean", "fit_s_mean", "pred_s_mean", "size_MB_mean"]].join(qual)
display(comp.sort_values("test_RMSE_mean").round(2))

## Aşama 9 — Aday Model Seçimi
### 9.1 Seçim ölçütleri ve öncelik sırası

Yönerge, seçimin yalnızca tek bir metriğe dayanmamasını istiyor. Problemin bağlamına göre ölçütler şu öncelik sırasıyla değerlendirildi:

| Öncelik | Ölçüt | Neden bu sırada? |
|---|---|---|
| 1 | **`test_RMSE`** (karar ufku) | Proje başarı kriteri ve literatürle karşılaştırma bu metrikle yapılıyor |
| 2 | **`test_NASA` + `late10_pct`** | Havacılık bakımında ömrü fazla tahmin etmek güvenlik riski; RMSE bu asimetriyi görmez |
| 3 | **`near50_RMSE`** | Bakım kararı arızaya yakın dönemde verilir; bu bölgedeki doğruluk işin özü |
| 4 | **Kararlılık** (fold std) | 100 motorluk küçük filoda tutarlı model, resmi teste daha güvenilir genellenir |
| 5 | **Maliyet** (eğitim/tahmin süresi, boyut) | Tuning bütçesi (Aşama 10) ve Streamlit Cloud'da deploy (Aşama 14) |
| 6 | **Yorumlanabilirlik** | Mühendisin tahmine güvenmesi için özellik önemi/SHAP sunulabilmeli |

### 9.2 Çok ölçütlü sıralama

In [ ]:
crit = summary.loc[list(models.keys())].copy()
crit["kararlılık (RMSE std)"] = crit["test_RMSE_std"]
rank_cols = {"test_RMSE_mean": "1-RMSE", "test_NASA_mean": "2a-NASA", "late10_pct_mean": "2b-geç tahmin %",
             "near50_RMSE_mean": "3-RUL≤50 RMSE", "kararlılık (RMSE std)": "4-kararlılık",
             "fit_s_mean": "5a-eğitim süresi", "size_MB_mean": "5b-boyut"}
ranks = crit[list(rank_cols)].rank(method="min").rename(columns=rank_cols).astype(int)
ranks["ortalama sıra"] = ranks.mean(axis=1).round(2)
display(ranks.sort_values(["1-RMSE"]))

*(1 = o ölçütte en iyi model.)* HistGradientBoosting, ilk beş ölçütün (isabet, NASA, geç tahmin, kritik bölge, kararlılık) **hepsinde 1. sırada**; yalnızca eğitim süresi ve boyutta doğrusal modellerin gerisinde ama yine de çok ucuz bir model. Ağırlıklı bir puanlama yerine sıra tablosu tercih edildi; ağırlıkları keyfi seçmek sonucu istenen yöne çekmeye açık olurdu. Öncelik sırası 9.1'de açıkça belirtildi.

### 9.3 Karar

**Aday model: `HistGradientBoostingRegressor`**

**Diğer modellerden neden ayrışıyor?**
1. **İsabet (öncelik 1):** En düşük test-benzeri RMSE (~16.0). Baseline'a göre ~%31 iyileşme; tuning yapılmadan bile proje hedef bandında (15–18).
2. **Güvenlik (öncelik 2):** Hem en düşük NASA skoru (~1030) hem de en düşük geç tahmin oranı (~%25; baseline'da ~%45). Yani hatalarının daha büyük kısmı temkinli tarafta.
3. **Kritik bölge (öncelik 3):** Arızaya yakın bölgede (RUL ≤ 50) en düşük hata (~8.9 RMSE).
4. **Kararlılık (öncelik 4):** Fold'lar arası en düşük standart sapma (~1.1); en yakın rakiplerini 5 fold'un çoğunda geçiyor.
5. **Maliyet (öncelik 5):** Fold başına ~1.5 sn eğitim, ~0.5 MB model. Random Forest'tan ~20 kat hızlı ve ~170 kat küçük. Bu, Aşama 10'da Optuna ile **geniş bir arama** yapabilmemizi ve Streamlit Cloud'da sorunsuz deploy etmemizi sağlıyor.
6. **İyileştirme potansiyeli:** 8.6'daki eğitim–validation farkı (~5.7 → ~13.2), düzenlileştirme parametrelerinin tuning ile iyileştirilebileceğini gösteriyor. Varsayılan ayarlarla en iyi olan bir modelin tuning'den de en çok faydalanması beklenir.
7. **Yorumlanabilirlik (öncelik 6):** Permutation importance ve SHAP ile hangi sensörlerin tahmini yönlendirdiği gösterilebilir.

**Problem bağlamıyla ilişkisi:** Motor bozulması, ömrün ikinci yarısında hızlanan ve sensörler arasında etkileşimli, **doğrusal olmayan** bir süreç (Bölüm 1, 3.5). Gradient boosting, bu tür eşik ve etkileşim yapılarını sığ ağaçların ardışık toplamıyla yakalamaya çok uygun. Doğrusal modellerin bu bölgede yetersiz öğrenmesi (8.6) ve sistematik geç tahmin yapması (8.7) da bunu doğruluyor.

**Değerlendirilip seçilmeyenler:**
- **Random Forest** (RMSE'de 2.): isabeti çok yakın (fark ~0.4) ama ~20 kat yavaş ve ~170 kat büyük; fold varyansı da daha yüksek. Tuning bütçesi ve deploy açısından pahalı.
- **SVR** (RMSE'de 3.): NASA skoru ikinci en iyi, ancak geç tahmin oranı (~%31) HGB'den yüksek. Tahmin süresi en uzun model ve eğitim maliyeti veriyle karesel büyüyor. Arızaya yakın bölgedeki hatası (~11.2) HGB'den (~8.9) belirgin şekilde yüksek.
- **MLP**: Fold'lar arası varyansı yüksek (kararsız) ve NASA skoru yüksek. Küçük veri setinde sinir ağının avantajı görülmüyor.

### 9.4 Sonraki aşamalar için plan

| Aşama | Model(ler) | Gerekçe |
|---|---|---|
| **10 — Hyperparameter optimization** | **HistGradientBoosting** (aday) | Optuna (Bayesian) + karşılaştırma için Random Search; `window` da arama uzayına eklenecek |
| **11 — Strong competitor comparison** | **Random Forest** ve **SVR** (Aşama 8'in 2. ve 3. modelleri, farklı aileler: bagging ve kernel) + **LightGBM/XGBoost** (aday ile aynı aileden güçlü kütüphaneler) | Yönergenin istediği gibi rakipler varsayılan ayarlarla bırakılmayacak; **aynı tuning bütçesi** ve aynı CV ile optimize edilecek |
| **12 — Final evaluation** | Aşama 11'in kazananı | Resmi test seti (100 motor) yalnızca bir kez açılacak |

### Sonuçları kaydet

In [ ]:
config.update({"candidate_model": "HistGradientBoostingRegressor",
               "strong_competitors": ["RandomForestRegressor", "SVR", "LightGBM/XGBoost"],
               "stage8_best_test_RMSE": round(float(summary.loc["HistGradientBoosting", "test_RMSE_mean"]), 3)})
json.dump(config, open("project_config.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps({k: config[k] for k in ["cap", "window", "candidate_model", "strong_competitors", "baseline_test_RMSE", "stage8_best_test_RMSE"] if k in config}, indent=2, ensure_ascii=False))
print("\nKaydedilen dosyalar:", [f for f in os.listdir(".") if f.startswith(("results_stage8", "project_config"))])